# Data Packaging for Pretraining

## Objectives
By the end of this demo you will be able to:
1. **Load** the cleaned pretraining dataset saved previously.
2. **Shard** a large dataset into manageable chunks for distributed training.
3. **Tokenize** raw text into token IDs using a pretrained tokenizer, adding BOS/EOS markers.
4. **Pack** variable-length tokenized sequences into fixed-length blocks — the standard format for pretraining.
5. **Save** the packed dataset to disk as Parquet for use in the training loop.

## Description
After data cleaning, the next step in the LLM pretraining pipeline is **data packaging**.
Raw text must be converted into integer token IDs, then packed into fixed-length sequences
that match the model's context window. This avoids wasted padding and maximises GPU utilisation.

**Tokenizer used:** `gpt2` — a classic BPE tokenizer, publicly available on Hugging Face,
conceptually identical to tokenizers used in modern LLMs (LLaMA, Mistral, GPT-4)

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Uncomment if not already installed
!pip install datasets transformers -q

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Tokenizing and Creating input_ids

### 1a. Load the Cleaned Dataset

We load the Parquet file produced by the data cleaning done previously


In [4]:
import datasets

dataset = datasets.load_dataset(
    "parquet",
    data_files="/content/drive/MyDrive/Colab Notebooks/data/preprocessed_dataset.parquet",
    split="train"
)
print(dataset)


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 4418
})


### 1b. Shard the Dataset

**Sharding** splits the dataset into N equal-sized pieces.
In a real distributed training job, each GPU worker processes one shard independently.
Here we use shard 0 (the first piece) to keep the demo fast.

>  For full pretraining runs, each shard is processed in parallel
> across multiple nodes. Sharding also lets you resume training from a checkpoint without reprocessing the entire corpus.


In [5]:
# Split into 10 shards, use only the first one for this demo
dataset = dataset.shard(num_shards=10, index=0)
print(f"Shard size: {dataset.num_rows:,} rows")
print(dataset)


Shard size: 442 rows
Dataset({
    features: ['text'],
    num_rows: 442
})


### 1c. Load the Tokenizer

We use the **GPT-2 tokenizer** — a Byte-Pair Encoding (BPE) tokenizer that is:
- Freely available on Hugging Face (no authentication needed)
- Conceptually identical to tokenizers in LLaMA, Mistral, and GPT-4
- Vocabulary size: 50,257 tokens

In [6]:
from transformers import AutoTokenizer

#tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")


# Inspect the tokenizer
print(f"Tokenizer: SmolLM2  |  Vocab size: {tokenizer.vocab_size}")
print(f"BOS token       : '{tokenizer.bos_token}' (id={tokenizer.bos_token_id})")
print(f"EOS token       : '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")


Tokenizer: SmolLM2  |  Vocab size: 49152
BOS token       : '<|endoftext|>' (id=0)
EOS token       : '<|endoftext|>' (id=0)


In [7]:
# Try the tokenizer on a sample sentence
sample_text = "I'm a short sentence"
tokens = tokenizer.tokenize(sample_text)
print("Tokens   :", tokens)
print("Token IDs:", tokenizer.convert_tokens_to_ids(tokens))


Tokens   : ['I', "'m", 'Ġa', 'Ġshort', 'Ġsentence']
Token IDs: [57, 5248, 253, 1890, 6330]


### 1d. Tokenize the Dataset

We define a `tokenization` function that:
1. Tokenizes each text document into subword tokens
2. Converts tokens to integer IDs
3. Prepends a **BOS** (Beginning Of Sequence) token and appends an **EOS** (End Of Sequence) token
4. Records `num_tokens` for later statistics

>  **BOS/EOS tokens** tell the model where a document starts and ends within a packed sequence.
> Without them, the model cannot distinguish document boundaries.


In [8]:
def tokenization(example):
    # Tokenize the text into subword tokens
    tokens = tokenizer.tokenize(example["text"])

    # Convert tokens to integer IDs
    token_ids = tokenizer.convert_tokens_to_ids(tokens)

    # Wrap with BOS (beginning of sequence) and EOS (end of sequence) markers
    token_ids = [tokenizer.bos_token_id] + token_ids + [tokenizer.eos_token_id]

    example["input_ids"] = token_ids
    example["num_tokens"] = len(token_ids)
    return example


In [9]:
# Apply tokenization to the full shard
dataset = dataset.map(tokenization, load_from_cache_file=False)
print(dataset)


Map:   0%|          | 0/442 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'num_tokens'],
    num_rows: 442
})


In [10]:
# Inspect a tokenized sample
sample = dataset[3]
print("Text (first 60 chars) :", sample["text"][:60])
print("input_ids (first 15)  :", sample["input_ids"][:15])
print("num_tokens            :", sample["num_tokens"])


Text (first 60 chars) :  As with previous Valkyira Chronicles games , Valkyria Chron
input_ids (first 15)  : [0, 1032, 351, 2672, 717, 1727, 105, 3792, 34471, 4252, 3297, 717, 1727, 105, 5484]
num_tokens            : 236


Check the total number of tokens in the shard — this tells us how much data we have:


In [11]:
import numpy as np

total_tokens = np.sum(dataset["num_tokens"])
print(f"Total tokens in shard: {total_tokens:,}")
print(f"Approx. equivalent to: {total_tokens / 1e6:.2f}M tokens")


Total tokens in shard: 69,185
Approx. equivalent to: 0.07M tokens


## 2. Packing the Data

### Why Pack?

Pretraining uses **fixed-length input sequences** (the model's context window, e.g. 512, 2048, 4096 tokens).
Documents vary widely in length — some are 50 tokens, others 2000.

**Naive approach:** pad each document to `max_seq_length` → wastes compute on padding tokens.

**Packing approach:** concatenate all token IDs into one long stream, then slice into
fixed-length chunks. This achieves ~100% GPU utilisation with zero wasted tokens.




>  `max_seq_length = 32` here for visibility. Real models use 512–8192.
> The principle is identical regardless of length.


In [12]:
# Concatenate all input_ids from every document into one long flat array
input_ids = np.concatenate(dataset["input_ids"])
print(f"Total token stream length: {len(input_ids):,}")

Total token stream length: 69,185


In [13]:
# Context window size — set small for demo clarity
# In practice: 512, 1024, 2048, 4096, 8192
max_seq_length = 32
print(f"Packing into chunks of {max_seq_length} tokens")


Packing into chunks of 32 tokens


In [14]:
# Trim to a length exactly divisible by max_seq_length
# (drop the last partial chunk — it would be incomplete)
total_length = len(input_ids) - len(input_ids) % max_seq_length
print(f"Trimmed length : {total_length:,}")
print(f"Tokens dropped : {len(input_ids) - total_length}")


Trimmed length : 69,184
Tokens dropped : 1


In [15]:
input_ids = input_ids[:total_length]
print(f"Shape after trim: {input_ids.shape}")


Shape after trim: (69184,)


In [16]:
# Reshape into (num_chunks, max_seq_length)
input_ids_reshaped = input_ids.reshape(-1, max_seq_length).astype(np.int32)
print(f"Packed shape: {input_ids_reshaped.shape}")
print(f"→ {input_ids_reshaped.shape[0]:,} training sequences of {max_seq_length} tokens each")


Packed shape: (2162, 32)
→ 2,162 training sequences of 32 tokens each


In [17]:
# Peek at the first packed sequence
print("First packed sequence (token IDs):")
print(input_ids_reshaped[0])
print("\nDecoded:")
print(tokenizer.decode(input_ids_reshaped[0].tolist()))


First packed sequence (token IDs):
[    0  6014    90 15037   787   717  1727   105  5484   216    35  1577
   810 42228 34471   365  4931  1577 17097   226   116   176   250   129
 26453 11100   129 10391   111 11100   121 10391]

Decoded:
<|endoftext|> Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァル�


Notice the decoded output contains **multiple sentence fragments** — this is expected!
Packing deliberately concatenates documents. The BOS/EOS tokens mark boundaries.

Convert the numpy array to a Hugging Face Dataset:


In [18]:
input_ids_list = input_ids_reshaped.tolist()
packaged_pretrain_dataset = datasets.Dataset.from_dict(
    {"input_ids": input_ids_list}
)
print(packaged_pretrain_dataset)


Dataset({
    features: ['input_ids'],
    num_rows: 2162
})


## 3. Save the Packed Dataset to Disk

Save as Parquet — this file is the direct input to the pretraining loop


In [19]:
#import os
#os.makedirs("./data", exist_ok=True)
#packaged_pretrain_dataset.to_parquet("./data/packaged_pretrain_dataset.parquet")
#print("Saved to: ./data/packaged_pretrain_dataset.parquet")


In [20]:
import os
# Define the path to save the dataset in Google Drive
drive_data_dir = '/content/drive/MyDrive/Colab Notebooks/data'
os.makedirs(drive_data_dir, exist_ok=True)
file_path_drive = os.path.join(drive_data_dir, 'packaged_pretrain_dataset.parquet')

dataset.to_parquet(file_path_drive)
print(f"Saved to: {file_path_drive}")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved to: /content/drive/MyDrive/Colab Notebooks/data/packaged_pretrain_dataset.parquet


In [21]:
# Reload and verify
verify = datasets.load_dataset(
    "parquet",
    #data_files="/data/packaged_pretrain_dataset.parquet",
     data_files="/content/drive/MyDrive/Colab Notebooks/data/packaged_pretrain_dataset.parquet",
    split="train"
)
print(f"Verified: {verify.num_rows:,} packed sequences of length {len(verify[0]['input_ids'])}")


Generating train split: 0 examples [00:00, ? examples/s]

Verified: 442 packed sequences of length 188


| Concept | Summary |
|---|---|
| Sharding | Splits dataset for parallel distributed processing |
| BPE Tokenizer | Converts text → subword tokens → integer IDs |
| BOS / EOS tokens | Mark document start/end within packed sequences |
| `num_tokens` | Used to audit total corpus size in tokens |
| Packing | Concatenate all tokens → slice into fixed-length chunks |
| `max_seq_length` | Must match the model's context window at training time |
| Zero padding | Packing achieves ~100% GPU utilisation, no wasted tokens |
